# 04 - Forecasting Models

## Objective

Evaluate monthly revenue forecasts using a chronological train/test split. The notebook compares Naive Forecast, Holt-Winters, ARIMA, and Auto-SARIMA models under different trend and seasonality assumptions.

In [ ]:
from pathlib import Path
import pandas as pd

# Caminho of the project
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "interm"

# Load the data
csv_file = DATA_PATH / "data_3.csv"
banco = pd.read_csv(csv_file)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Parâmetros of estilo of the Gráfico based in the manual of the MBA
plt.rcParams.update({

    # font
    "font.family": "Arial",
    "font.size": 11,

    # background
    "figure.facecolor": "white",
    "axes.facecolor": "white",

    # without grid
    "axes.grid": False,

    # without title
    "axes.titlesize": 0,

    # axes
    "axes.edgecolor": "black",
    "axes.linewidth": 1.5,

    # color of the textos
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",

    # size of the titles of thexes
    "axes.labelsize": 11

})

In [ ]:
def build_monthly_faturamento_total(df, start='2017-01-01', end='2025-08-01'):
    """Returns a DataFrame indexed by month with continuous monthly revenue."""
    # Use existing calendar columns when available; otherwise extract them from data_venda
    if 'ano' in df.columns and 'mes' in df.columns:
        tmp = (
            df.groupby(['ano', 'mes'])['valor_total']
            .sum()
            .reset_index()
        )
        tmp['date'] = pd.to_datetime(dict(year=tmp['ano'], month=tmp['mes'], day=1))
    else:
        tmp = (
            df.groupby([df['data_venda'].dt.year, df['data_venda'].dt.month])['valor_total']
            .sum()
            .reset_index(name='valor_total')
        )
        tmp.columns = ['ano', 'mes', 'valor_total']
        tmp['date'] = pd.to_datetime(dict(year=tmp['ano'], month=tmp['mes'], day=1))
    monthly = tmp.sort_values('date').set_index('date')[['valor_total']]
    idx = pd.date_range(start=start, end=end, freq='MS')
    monthly = monthly.reindex(idx, fill_value=0)
    monthly.index.name = 'date'
    return monthly

In [ ]:
def plot_with_style(
    data,
    kind='line',                 # 'line' or 'bar'
    x_label='',
    y_label='',
    title='',
    xticks=None,                 # custom ticks
    xtlabels=None,               # labels personalizados
    rotate_xticks=0,
    figsize=(12, 6),
    marker='o',
    linewidth=2,
    colors=None,
    linestyles=None,
    xlim=None,                   # e.g., ('2017-01-01', '2026-01-01')
    year_ticks=True,             # ticks annual automáticos
    vline=None,                  # e.g., test.index[0]
    vline_label=None
):

    # ---------------------------------------------------------
    # Validation
    # ---------------------------------------------------------

    if not isinstance(data, (pd.Series, pd.DataFrame)):
        raise ValueError(
            'date must be pd.Series or pd.DataFrame'
        )

    if kind not in ['line', 'bar']:
        raise ValueError(
            "kind must be 'line' or 'bar'"
        )

    # ---------------------------------------------------------
    # Create the figure
    # ---------------------------------------------------------

    fig, ax = plt.subplots(figsize=figsize)

    # ---------------------------------------------------------
    # Gráfico of Series
    # ---------------------------------------------------------

    if isinstance(data, pd.Series):

        label = data.name if data.name is not None else 'Series'

        if kind == 'line':

            ax.plot(
                data.index,
                data.values,
                marker=marker,
                linewidth=linewidth,
                label=label
            )

        else:

            ax.bar(
                data.index,
                data.values,
                edgecolor='black',
                linewidth=0.8,
                label=label
            )

    # ---------------------------------------------------------
    # Gráfico of DataFrame
    # ---------------------------------------------------------

    elif isinstance(data, pd.DataFrame):

        if kind == 'line':

            for i, column in enumerate(data.columns):

                kwargs = {
                    'marker': marker,
                    'linewidth': linewidth,
                    'label': column
                }

                if colors is not None:
                    kwargs['color'] = colors[i]

                if linestyles is not None:
                    kwargs['linestyle'] = linestyles[i]

                ax.plot(
                    data.index,
                    data[column],
                    **kwargs
                )

        else:

            data.plot(
                kind='bar',
                ax=ax
            )

    # ---------------------------------------------------------
    # labels
    # ---------------------------------------------------------

    ax.set_xlabel(
        x_label,
        fontsize=11,
        fontname='Arial'
    )

    ax.set_ylabel(
        y_label,
        fontsize=11,
        fontname='Arial'
    )

    ax.set_title(
        title,
        fontsize=13,
        fontname='Arial'
    )

    # ---------------------------------------------------------
    # Estilo of thexes
    # ---------------------------------------------------------

    ax.grid(False)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.spines['left'].set_visible(True)
    ax.spines['bottom'].set_visible(True)

    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)

    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_color('black')

    # ---------------------------------------------------------
    # Limites of thexis X
    # ---------------------------------------------------------

    if xlim is not None:

        if isinstance(data.index, pd.DatetimeIndex):

            xlim = (
                pd.to_datetime(xlim[0]),
                pd.to_datetime(xlim[1])
            )

        ax.set_xlim(xlim)

    # ---------------------------------------------------------
    # Ticks temporais annual
    # ---------------------------------------------------------

    if (
        year_ticks
        and isinstance(data.index, pd.DatetimeIndex)
        and xticks is None
    ):

        ax.xaxis.set_major_locator(
            mdates.YearLocator()
        )

        ax.xaxis.set_major_formatter(
            mdates.DateFormatter('%Y')
        )

    # ---------------------------------------------------------
    # Ticks personalizados
    # ---------------------------------------------------------

    if xticks is not None:

        ax.set_xticks(xticks)

    if xtlabels is not None:

        ax.set_xticklabels(
            xtlabels,
            fontname='Arial'
        )

    # ---------------------------------------------------------
    # Tick rotation
    # ---------------------------------------------------------

    if rotate_xticks:

        plt.setp(
            ax.get_xticklabels(),
            rotation=rotate_xticks,
            ha='right'
        )

    # ---------------------------------------------------------
    # Vertical reference line (ex.: Start of test period)
    # ---------------------------------------------------------

    if vline is not None:

        ax.axvline(
            vline,
            linestyle='--',
            linewidth=1.5,
            label=vline_label,
            c='r'
        )

    # ---------------------------------------------------------
    # legend
    # ---------------------------------------------------------

    handles, labels = ax.get_legend_handles_labels()

    if len(labels) > 1:
        ax.legend(
            title=None,
            frameon=False
        )

    # ---------------------------------------------------------
    # Layout
    # ---------------------------------------------------------

    plt.tight_layout()

    plt.show()

In [ ]:
faturamento_mensal = build_monthly_faturamento_total(banco, start='2017-02-01', end='2025-08-31')['valor_total']

In [ ]:
plot_with_style(
    faturamento_mensal,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True
)

## Training and Test Split

Because the data is a time series, a random `train_test_split` would leak future information into the training set. The first 80% of observations are used for training and the final 20% for testing.

In [ ]:
train_size = int(len(faturamento_mensal) * 0.80)

train = faturamento_mensal.iloc[:train_size].copy()
test = faturamento_mensal.iloc[train_size:].copy()

## Naive Forecast Baseline

The Naive model uses the previous observation as the forecast. It provides a simple baseline for evaluating more complex methods.

In [ ]:
pred_naive = test.shift(1)
pred_naive.iloc[0] = train.iloc[-1]

In [ ]:
mae_naive = mean_absolute_error(test, pred_naive)

rmse_naive = np.sqrt(
    mean_squared_error(test, pred_naive)
)

print(f"MAE - Naive: {mae_naive:.2f}")
print(f"RMSE - Naive: {rmse_naive:.2f}")

In [ ]:
grafico_naive = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Naive': pred_naive
})

In [ ]:
plot_with_style(
    grafico_naive,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

---

## Holt-Winters Exponential Smoothing

### 01 - Additive Trend and No Seasonality

In [ ]:
modelo_hw_01 = ExponentialSmoothing(
    train,
    trend="add", # trend additive
    seasonal=None # seasonality none
)

ajuste_hw_01 = modelo_hw_01.fit()

pred_hw_01 = ajuste_hw_01.forecast(len(test))

In [ ]:
mae_hw_01 = mean_absolute_error(
    test,
    pred_hw_01
)

rmse_hw_01 = np.sqrt(
    mean_squared_error(test, pred_hw_01)
)

print(f"MAE - Holt-Winters (Additive Trend and No Seasonality): {mae_hw_01:.2f}")
print(f"RMSE - Holt-Winters (Additive Trend and No Seasonality): {rmse_hw_01:.2f}")

In [ ]:
grafico_hw_01 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend additive and seasonality none)': pred_hw_01
})

In [ ]:
plot_with_style(
    grafico_hw_01,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

### 02 - Multiplicative Trend and No Seasonality

In [ ]:
modelo_hw_02 = ExponentialSmoothing(
    train,
    trend="mul", # trend multiplicative
    seasonal=None # seasonality none
)

ajuste_hw_02 = modelo_hw_02.fit()

pred_hw_02 = ajuste_hw_02.forecast(len(test))

In [ ]:
mae_hw_02 = mean_absolute_error(
    test,
    pred_hw_02
)

rmse_hw_02 = np.sqrt(
    mean_squared_error(test, pred_hw_02)
)

print(f"MAE - Holt-Winters (Multiplicative Trend and No Seasonality): {mae_hw_02:.2f}")
print(f"RMSE - Holt-Winters (Multiplicative Trend and No Seasonality): {rmse_hw_02:.2f}")

In [ ]:
grafico_hw_02 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend multiplicative and seasonality none)': pred_hw_02
})

In [ ]:
plot_with_style(
    grafico_hw_02,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

### 03 - Additive Trend and Seasonality

#### Seasonal Period: 3 Months

In [ ]:
modelo_hw_03 = ExponentialSmoothing(
    train,
    trend="add", # trend additive
    seasonal="add", # seasonality additive
    seasonal_periods=3 # Seasonal period of 3 months (1 trimestre)
)

ajuste_hw_03 = modelo_hw_03.fit()

pred_hw_03 = ajuste_hw_03.forecast(len(test))

In [ ]:
mae_hw_03 = mean_absolute_error(
    test,
    pred_hw_03
)

rmse_hw_03 = np.sqrt(
    mean_squared_error(test, pred_hw_03)
)

print(f"MAE - Holt-Winters (Additive Trend and Seasonality): {mae_hw_03:.2f}")
print(f"RMSE - Holt-Winters (Additive Trend and Seasonality): {rmse_hw_03:.2f}")

In [ ]:
grafico_hw_03 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend and seasonality additive)': pred_hw_03
})

In [ ]:
plot_with_style(
    grafico_hw_03,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

#### Seasonal Period: 5 Months

In [ ]:
modelo_hw_03 = ExponentialSmoothing(
    train,
    trend="add", # trend additive
    seasonal="add", # seasonality additive
    seasonal_periods=5 # Seasonal period of 5 months
)

ajuste_hw_03 = modelo_hw_03.fit()

pred_hw_03 = ajuste_hw_03.forecast(len(test))

In [ ]:
mae_hw_03 = mean_absolute_error(
    test,
    pred_hw_03
)

rmse_hw_03 = np.sqrt(
    mean_squared_error(test, pred_hw_03)
)

print(f"MAE - Holt-Winters (Additive Trend and Seasonality): {mae_hw_03:.2f}")
print(f"RMSE - Holt-Winters (Additive Trend and Seasonality): {rmse_hw_03:.2f}")

In [ ]:
grafico_hw_03 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend and seasonality additive)': pred_hw_03
})

In [ ]:
plot_with_style(
    grafico_hw_03,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

#### Seasonal Period: 6 Months

In [ ]:
modelo_hw_03 = ExponentialSmoothing(
    train,
    trend="add", # trend additive
    seasonal="add", # seasonality additive
    seasonal_periods=6 # Seasonal period of 6 months (2 trimestres)
)

ajuste_hw_03 = modelo_hw_03.fit()

pred_hw_03 = ajuste_hw_03.forecast(len(test))

In [ ]:
mae_hw_03 = mean_absolute_error(
    test,
    pred_hw_03
)

rmse_hw_03 = np.sqrt(
    mean_squared_error(test, pred_hw_03)
)

print(f"MAE - Holt-Winters (Additive Trend and Seasonality): {mae_hw_03:.2f}")
print(f"RMSE - Holt-Winters (Additive Trend and Seasonality): {rmse_hw_03:.2f}")

In [ ]:
grafico_hw_03 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend and seasonality additive)': pred_hw_03
})

In [ ]:
plot_with_style(
    grafico_hw_03,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

#### Seasonal Period: 12 Months

In [ ]:
modelo_hw_03 = ExponentialSmoothing(
    train,
    trend="add", # trend additive
    seasonal="add", # seasonality additive
    seasonal_periods=12 # Seasonal period of 12 months (4 trimestres)
)

ajuste_hw_03 = modelo_hw_03.fit()

pred_hw_03 = ajuste_hw_03.forecast(len(test))

In [ ]:
mae_hw_03 = mean_absolute_error(
    test,
    pred_hw_03
)

rmse_hw_03 = np.sqrt(
    mean_squared_error(test, pred_hw_03)
)

print(f"MAE - Holt-Winters (Additive Trend and Seasonality): {mae_hw_03:.2f}")
print(f"RMSE - Holt-Winters (Additive Trend and Seasonality): {rmse_hw_03:.2f}")

In [ ]:
grafico_hw_03 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend and seasonality additive)': pred_hw_03
})

In [ ]:
plot_with_style(
    grafico_hw_03,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

### 04 - Additive Trend and Multiplicative Seasonality

#### Seasonal Period: 3 Months

In [ ]:
modelo_hw_04 = ExponentialSmoothing(
    train,
    trend="add", # trend additive
    seasonal="mul", # seasonality multiplicative
    seasonal_periods=3 # Seasonal period of 3 months (1 trimestre)
)

ajuste_hw_04 = modelo_hw_04.fit()

pred_hw_04 = ajuste_hw_04.forecast(len(test))

In [ ]:
mae_hw_04 = mean_absolute_error(
    test,
    pred_hw_04
)

rmse_hw_04 = np.sqrt(
    mean_squared_error(test, pred_hw_04)
)

print(f"MAE - Holt-Winters (Additive Trend and Multiplicative Seasonality): {mae_hw_04:.2f}")
print(f"RMSE - Holt-Winters (Additive Trend and Multiplicative Seasonality): {rmse_hw_04:.2f}")

In [ ]:
grafico_hw_04 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend additive and seasonality multiplicative)': pred_hw_04
})

In [ ]:
plot_with_style(
    grafico_hw_04,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

#### Seasonal Period: 5 Months

In [ ]:
modelo_hw_04 = ExponentialSmoothing(
    train,
    trend="add", # trend additive
    seasonal="mul", # seasonality multiplicative
    seasonal_periods=5 # Seasonal period of 5 months
)

ajuste_hw_04 = modelo_hw_04.fit()

pred_hw_04 = ajuste_hw_04.forecast(len(test))

In [ ]:
mae_hw_04 = mean_absolute_error(
    test,
    pred_hw_04
)

rmse_hw_04 = np.sqrt(
    mean_squared_error(test, pred_hw_04)
)

print(f"MAE - Holt-Winters (Additive Trend and Multiplicative Seasonality): {mae_hw_04:.2f}")
print(f"RMSE - Holt-Winters (Additive Trend and Multiplicative Seasonality): {rmse_hw_04:.2f}")

In [ ]:
grafico_hw_04 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend additive and seasonality multiplicative)': pred_hw_04
})

In [ ]:
plot_with_style(
    grafico_hw_04,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

#### Seasonal Period: 6 Months

In [ ]:
modelo_hw_04 = ExponentialSmoothing(
    train,
    trend="add", # trend additive
    seasonal="mul", # seasonality multiplicative
    seasonal_periods=6 # Seasonal period of 6 months (2 trimestres)
)

ajuste_hw_04 = modelo_hw_04.fit()

pred_hw_04 = ajuste_hw_04.forecast(len(test))

In [ ]:
mae_hw_04 = mean_absolute_error(
    test,
    pred_hw_04
)

rmse_hw_04 = np.sqrt(
    mean_squared_error(test, pred_hw_04)
)

print(f"MAE - Holt-Winters (Additive Trend and Multiplicative Seasonality): {mae_hw_04:.2f}")
print(f"RMSE - Holt-Winters (Additive Trend and Multiplicative Seasonality): {rmse_hw_04:.2f}")

In [ ]:
grafico_hw_04 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend additive and seasonality multiplicative)': pred_hw_04
})

In [ ]:
plot_with_style(
    grafico_hw_04,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

#### Seasonal Period: 12 Months

In [ ]:
modelo_hw_04 = ExponentialSmoothing(
    train,
    trend="add", # trend additive
    seasonal="mul", # seasonality multiplicative
    seasonal_periods=12 # Seasonal period of 12 months (1 year)
)

ajuste_hw_04 = modelo_hw_04.fit()

pred_hw_04 = ajuste_hw_04.forecast(len(test))

In [ ]:
mae_hw_04 = mean_absolute_error(
    test,
    pred_hw_04
)

rmse_hw_04 = np.sqrt(
    mean_squared_error(test, pred_hw_04)
)

print(f"MAE - Holt-Winters (Additive Trend and Multiplicative Seasonality): {mae_hw_04:.2f}")
print(f"RMSE - Holt-Winters (Additive Trend and Multiplicative Seasonality): {rmse_hw_04:.2f}")

In [ ]:
grafico_hw_04 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend additive and seasonality multiplicative)': pred_hw_04
})

In [ ]:
plot_with_style(
    grafico_hw_04,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

### 05 - Multiplicative Trend and Additive Seasonality

#### Seasonal Period: 3 Months

In [ ]:
modelo_hw_05 = ExponentialSmoothing(
    train,
    trend="mul", # trend multiplicative
    seasonal="add", # seasonality additive
    seasonal_periods=3 # Seasonal period of 3 months (1 trimestre)
)

ajuste_hw_05 = modelo_hw_05.fit()

pred_hw_05 = ajuste_hw_05.forecast(len(test))

In [ ]:
mae_hw_05 = mean_absolute_error(
    test,
    pred_hw_05
)

rmse_hw_05 = np.sqrt(
    mean_squared_error(test, pred_hw_05)
)

print(f"MAE - Holt-Winters (Multiplicative Trend and Additive Seasonality): {mae_hw_05:.2f}")
print(f"RMSE - Holt-Winters (Multiplicative Trend and Additive Seasonality): {rmse_hw_05:.2f}")

In [ ]:
grafico_hw_05 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend multiplicative and seasonality additive)': pred_hw_05
})

In [ ]:
plot_with_style(
    grafico_hw_05,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

#### Seasonal Period: 5 Months

In [ ]:
modelo_hw_05 = ExponentialSmoothing(
    train,
    trend="mul", # trend multiplicative
    seasonal="add", # seasonality additive
    seasonal_periods=5 # Seasonal period of 6 months
)

ajuste_hw_05 = modelo_hw_05.fit()

pred_hw_05 = ajuste_hw_05.forecast(len(test))

In [ ]:
mae_hw_05 = mean_absolute_error(
    test,
    pred_hw_05
)

rmse_hw_05 = np.sqrt(
    mean_squared_error(test, pred_hw_05)
)

print(f"MAE - Holt-Winters (Multiplicative Trend and Additive Seasonality): {mae_hw_05:.2f}")
print(f"RMSE - Holt-Winters (Multiplicative Trend and Additive Seasonality): {rmse_hw_05:.2f}")

In [ ]:
grafico_hw_05 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend multiplicative and seasonality additive)': pred_hw_05
})

In [ ]:
plot_with_style(
    grafico_hw_05,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

#### Seasonal Period: 6 Months

In [ ]:
modelo_hw_05 = ExponentialSmoothing(
    train,
    trend="mul", # trend multiplicative
    seasonal="add", # seasonality additive
    seasonal_periods=6 # Seasonal period of 6 months
)

ajuste_hw_05 = modelo_hw_05.fit()

pred_hw_05 = ajuste_hw_05.forecast(len(test))

In [ ]:
mae_hw_05 = mean_absolute_error(
    test,
    pred_hw_05
)

rmse_hw_05 = np.sqrt(
    mean_squared_error(test, pred_hw_05)
)

print(f"MAE - Holt-Winters (Multiplicative Trend and Additive Seasonality): {mae_hw_05:.2f}")
print(f"RMSE - Holt-Winters (Multiplicative Trend and Additive Seasonality): {rmse_hw_05:.2f}")

In [ ]:
grafico_hw_05 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend multiplicative and seasonality additive)': pred_hw_05
})

In [ ]:
plot_with_style(
    grafico_hw_05,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

#### Seasonal Period: 12 Months

In [ ]:
modelo_hw_05 = ExponentialSmoothing(
    train,
    trend="mul", # trend multiplicative
    seasonal="add", # seasonality additive
    seasonal_periods=12 # Seasonal period of 12 months (1 year)
)

ajuste_hw_05 = modelo_hw_05.fit()

pred_hw_05 = ajuste_hw_05.forecast(len(test))

In [ ]:
mae_hw_05 = mean_absolute_error(
    test,
    pred_hw_05
)

rmse_hw_05 = np.sqrt(
    mean_squared_error(test, pred_hw_05)
)

print(f"MAE - Holt-Winters (Multiplicative Trend and Additive Seasonality): {mae_hw_05:.2f}")
print(f"RMSE - Holt-Winters (Multiplicative Trend and Additive Seasonality): {rmse_hw_05:.2f}")

In [ ]:
grafico_hw_05 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend multiplicative and seasonality additive)': pred_hw_05
})

In [ ]:
plot_with_style(
    grafico_hw_05,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

### 06 - Multiplicative Trend and Seasonality

#### Seasonal Period: 3 Months

In [ ]:
modelo_hw_06 = ExponentialSmoothing(
    train,
    trend="mul", # trend multiplicative
    seasonal="mul", # seasonality multiplicative
    seasonal_periods=3 # Seasonal period of 3 months (1 trimestre)
)

ajuste_hw_06 = modelo_hw_06.fit()

pred_hw_06 = ajuste_hw_06.forecast(len(test))

In [ ]:
mae_hw_06 = mean_absolute_error(
    test,
    pred_hw_06
)

rmse_hw_06 = np.sqrt(
    mean_squared_error(test, pred_hw_06)
)

print(f"MAE - Holt-Winters (Multiplicative Trend and Seasonality): {mae_hw_06:.2f}")
print(f"RMSE - Holt-Winters (Multiplicative Trend and Seasonality): {rmse_hw_06:.2f}")

In [ ]:
grafico_hw_06 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend and seasonality multiplicative)': pred_hw_06
})

In [ ]:
plot_with_style(
    grafico_hw_06,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

#### Seasonal Period: 5 Months

In [ ]:
modelo_hw_06 = ExponentialSmoothing(
    train,
    trend="mul", # trend multiplicative
    seasonal="mul", # seasonality multiplicative
    seasonal_periods=5 # Seasonal period of 5 months
)

ajuste_hw_06 = modelo_hw_06.fit()

pred_hw_06 = ajuste_hw_06.forecast(len(test))

In [ ]:
mae_hw_06 = mean_absolute_error(
    test,
    pred_hw_06
)

rmse_hw_06 = np.sqrt(
    mean_squared_error(test, pred_hw_06)
)

print(f"MAE - Holt-Winters (Multiplicative Trend and Seasonality): {mae_hw_06:.2f}")
print(f"RMSE - Holt-Winters (Multiplicative Trend and Seasonality): {rmse_hw_06:.2f}")

In [ ]:
grafico_hw_06 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend and seasonality multiplicative)': pred_hw_06
})

In [ ]:
plot_with_style(
    grafico_hw_06,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

#### Seasonal Period: 6 Months

In [ ]:
modelo_hw_06 = ExponentialSmoothing(
    train,
    trend="mul", # trend multiplicative
    seasonal="mul", # seasonality multiplicative
    seasonal_periods=6 # Seasonal period of 6 months (2 trimestres)
)

ajuste_hw_06 = modelo_hw_06.fit()

pred_hw_06 = ajuste_hw_06.forecast(len(test))

In [ ]:
mae_hw_06 = mean_absolute_error(
    test,
    pred_hw_06
)

rmse_hw_06 = np.sqrt(
    mean_squared_error(test, pred_hw_06)
)

print(f"MAE - Holt-Winters (Multiplicative Trend and Seasonality): {mae_hw_06:.2f}")
print(f"RMSE - Holt-Winters (Multiplicative Trend and Seasonality): {rmse_hw_06:.2f}")

In [ ]:
grafico_hw_06 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend and seasonality multiplicative)': pred_hw_06
})

In [ ]:
plot_with_style(
    grafico_hw_06,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

#### Seasonal Period: 12 Months

In [ ]:
modelo_hw_06 = ExponentialSmoothing(
    train,
    trend="mul", # trend multiplicative
    seasonal="mul", # seasonality multiplicative
    seasonal_periods=12 # Seasonal period of 12 months (1 year)
)

ajuste_hw_06 = modelo_hw_06.fit()

pred_hw_06 = ajuste_hw_06.forecast(len(test))

In [ ]:
mae_hw_06 = mean_absolute_error(
    test,
    pred_hw_06
)

rmse_hw_06 = np.sqrt(
    mean_squared_error(test, pred_hw_06)
)

print(f"MAE - Holt-Winters (Multiplicative Trend and Seasonality): {mae_hw_06:.2f}")
print(f"RMSE - Holt-Winters (Multiplicative Trend and Seasonality): {rmse_hw_06:.2f}")

In [ ]:
grafico_hw_06 = pd.DataFrame({
    'training': train,
    'Actual': test,
    'Forecast - Holt-Winters (trend and seasonality multiplicative)': pred_hw_06
})

In [ ]:
plot_with_style(
    grafico_hw_06,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=test.index[0],
    vline_label='Start of test period'
)

---

## Manual ARIMA (p, d, q)

The target is `revenue_monthly`. The parameter `d` is the number of differences required for stationarity, `p` is the autoregressive order, and `q` is the moving-average order.

The workflow is:

1. Test stationarity with the Augmented Dickey-Fuller test.
2. Difference the series if necessary.
3. Use ACF and PACF to guide the selection of `p` and `q`.

### Stationarity Test: Augmented Dickey-Fuller (ADF)

- Null hypothesis (H0): the series has a unit root and is not stationary.
- Alternative hypothesis (H1): the series is stationary.
- If p < 0.05, reject H0 and conclude that the series is stationary.

In [ ]:
from statsmodels.tsa.stattools import adfuller

resultado_adf = adfuller(train)

print(f"ADF Statistic: {resultado_adf[0]:.4f}")
print(f"p-value: {resultado_adf[1]:.4f}")

if resultado_adf[1] < 0.05:
    print("The time series is stationary (reject H0).")
else:
    print("The time series is not stationary (fail to reject H0).")

First difference: $\Delta y_t = y_t - y_{t-1}$

In [ ]:
train_diff1 = train.diff().dropna()

In [ ]:
resultado_adf_diff1 = adfuller(train_diff1)

print(
    f"ADF Statistic: "
    f"{resultado_adf_diff1[0]:.4f}"
)

print(
    f"p-value: "
    f"{resultado_adf_diff1[1]:.4f}"
)
if resultado_adf_diff1[1] < 0.05:
    print("The time series is stationary (reject H0).")
else:
    print("The time series is not stationary (fail to reject H0).")

**Conclusion:** After first differencing, the series becomes stationary, so `d = 1` is used.

### Autocorrelation and Partial Autocorrelation

- `p`: use the PACF cutoff to guide the autoregressive order.
- `q`: use the ACF cutoff to guide the moving-average order.
- A significant lag lies outside the confidence interval.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

max_lag = 36

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ACF
plot_acf(
    train_diff1,
    lags=max_lag,
    ax=axes[0]
)
axes[0].set_xlabel(
    'Lag (months)',
    fontsize=11,
    fontname='Arial'
)
axes[0].set_ylabel(
    'Autocorrelation',
    fontsize=11,
    fontname='Arial'
)
axes[0].grid(False)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].spines['left'].set_linewidth(1.5)
axes[0].spines['bottom'].set_linewidth(1.5)
axes[0].spines['left'].set_color('black')
axes[0].spines['bottom'].set_color('black')
axes[0].set_xticks(list(range(0, max_lag + 1, 6)))
#axes[0].set_title(
#    'ACF of the Series Diferenciada',
#    fontsize=14,
#    fontweight='bold',
#    fontname='Arial'
#)
axes[0].text(
    0.95, 0.95, '(a)',
    transform=axes[0].transAxes,
    ha='right',
    va='top',
    fontsize=13,
    fontweight='bold',
    fontname='Arial'
)

# PACF
plot_pacf(
    train_diff1,
    lags=max_lag,
    ax=axes[1]
)
axes[1].set_xlabel(
    'Lag (months)',
    fontsize=11,
    fontname='Arial'
)
axes[1].set_ylabel(
    'Partial Autocorrelation',
    fontsize=11,
    fontname='Arial'
)
axes[1].grid(False)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].spines['left'].set_linewidth(1.5)
axes[1].spines['bottom'].set_linewidth(1.5)
axes[1].spines['left'].set_color('black')
axes[1].spines['bottom'].set_color('black')
axes[1].set_xticks(list(range(0, max_lag + 1, 6)))
#axes[1].set_title(
#    'PACF of the Series Diferenciada',
#    fontsize=14,
#    fontweight='bold',
#    fontname='Arial'
#)
axes[1].text(
    0.95, 0.95, '(b)',
    transform=axes[1].transAxes,
    ha='right',
    va='top',
    fontsize=13,
    fontweight='bold',
    fontname='Arial'
)

plt.tight_layout()
plt.show()

**Conclusion:** The ACF suggests `q = 1`, while the PACF suggests `p = 2`. The ARIMA(2,1,1) model is therefore compared with Auto-SARIMA.

### ARIMA(2,1,1)

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

modelo = ARIMA(
    train,
    order=(2,1,1)
)

ajuste_arima_211 = modelo.fit()

pred_arima_211 = ajuste_arima_211.forecast(
    steps=len(test)
)

In [ ]:
mae_arima_211 = mean_absolute_error(test, pred_arima_211)

rmse_arima_211 = np.sqrt(mean_squared_error(test, pred_arima_211))

print(f'AIC: {ajuste_arima_211.aic:.2f}')
print(f'MAE: {mae_arima_211:.2f}')
print(f'RMSE: {rmse_arima_211:.2f}')

In [ ]:
grafico_arima = pd.DataFrame({
    "training": train,
    "Actual": test,
    "Forecast - ARIMA(2,1,1)": pred_arima_211
})

plot_with_style(
    grafico_arima,
    x_label="year",
    y_label="revenue (R$)",
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=("2017-01-01", "2026-01-01"),
    year_ticks=True,
    vline=test.index[0],
    vline_label="Start of test period"
)

### Auto-ARIMA

#### Seasonal Period: 1 Month

In [ ]:
from pmdarima import auto_arima

modelo_arima = auto_arima(
    train,
    seasonal=True,
    m=1,
    stepwise=False,
    trace=True,
    suppress_warnings=True,
    error_action="ignore"
)

print(modelo_arima.summary())

In [ ]:
pred_auto_sarima = pd.Series(
    modelo_arima.predict(n_periods=len(test)),
    index=test.index
)

mae_auto_sarima = mean_absolute_error(test, pred_auto_sarima)
rmse_auto_sarima = np.sqrt(mean_squared_error(test, pred_auto_sarima))

print(f"MAE - Auto-SARIMA: {mae_auto_sarima:.2f}")
print(f"RMSE - Auto-SARIMA: {rmse_auto_sarima:.2f}")

#### Seasonal Period: 2 Months

In [ ]:
from pmdarima import auto_arima

modelo_arima = auto_arima(
    train,
    seasonal=True,
    m=2,
    stepwise=False,
    trace=True,
    suppress_warnings=True,
    error_action="ignore"
)

print(modelo_arima.summary())

In [ ]:
pred_auto_sarima = pd.Series(
    modelo_arima.predict(n_periods=len(test)),
    index=test.index
)

mae_auto_sarima = mean_absolute_error(test, pred_auto_sarima)
rmse_auto_sarima = np.sqrt(mean_squared_error(test, pred_auto_sarima))

print(f"MAE - Auto-SARIMA: {mae_auto_sarima:.2f}")
print(f"RMSE - Auto-SARIMA: {rmse_auto_sarima:.2f}")

#### Seasonal Period: 3 Months

In [ ]:
from pmdarima import auto_arima

modelo_arima = auto_arima(
    train,
    seasonal=True,
    m=3,
    stepwise=False,
    trace=True,
    suppress_warnings=True,
    error_action="ignore"
)

print(modelo_arima.summary())

In [ ]:
pred_auto_sarima = pd.Series(
    modelo_arima.predict(n_periods=len(test)),
    index=test.index
)

mae_auto_sarima = mean_absolute_error(test, pred_auto_sarima)
rmse_auto_sarima = np.sqrt(mean_squared_error(test, pred_auto_sarima))

print(f"MAE - Auto-SARIMA: {mae_auto_sarima:.2f}")
print(f"RMSE - Auto-SARIMA: {rmse_auto_sarima:.2f}")

In [ ]:
grafico_auto_sarima = pd.DataFrame({
    "training": train,
    "Actual": test,
    "Forecast - Auto-SARIMA": pred_auto_sarima
})

plot_with_style(
    grafico_auto_sarima,
    x_label="year",
    y_label="revenue (R$)",
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=("2017-01-01", "2026-01-01"),
    year_ticks=True,
    vline=test.index[0],
    vline_label="Start of test period"
)

#### Seasonal Period: 4 Months

In [ ]:
from pmdarima import auto_arima

modelo_arima = auto_arima(
    train,
    seasonal=True,
    m=4,
    stepwise=False,
    trace=True,
    suppress_warnings=True,
    error_action="ignore"
)

print(modelo_arima.summary())

In [ ]:
pred_auto_sarima = pd.Series(
    modelo_arima.predict(n_periods=len(test)),
    index=test.index
)

mae_auto_sarima = mean_absolute_error(test, pred_auto_sarima)
rmse_auto_sarima = np.sqrt(mean_squared_error(test, pred_auto_sarima))

print(f"MAE - Auto-SARIMA: {mae_auto_sarima:.2f}")
print(f"RMSE - Auto-SARIMA: {rmse_auto_sarima:.2f}")

#### Seasonal Period: 5 Months

In [ ]:
from pmdarima import auto_arima

modelo_arima = auto_arima(
    train,
    seasonal=True,
    m=5,
    stepwise=False,
    trace=True,
    suppress_warnings=True,
    error_action="ignore"
)

print(modelo_arima.summary())

In [ ]:
pred_auto_sarima = pd.Series(
    modelo_arima.predict(n_periods=len(test)),
    index=test.index
)

mae_auto_sarima = mean_absolute_error(test, pred_auto_sarima)
rmse_auto_sarima = np.sqrt(mean_squared_error(test, pred_auto_sarima))

print(f"MAE - Auto-SARIMA: {mae_auto_sarima:.2f}")
print(f"RMSE - Auto-SARIMA: {rmse_auto_sarima:.2f}")

#### Seasonal Period: 6 Months

In [ ]:
from pmdarima import auto_arima

modelo_arima = auto_arima(
    train,
    seasonal=True,
    m=6,
    stepwise=False,
    trace=True,
    suppress_warnings=True,
    error_action="ignore"
)

print(modelo_arima.summary())

In [ ]:
pred_auto_sarima = pd.Series(
    modelo_arima.predict(n_periods=len(test)),
    index=test.index
)

mae_auto_sarima = mean_absolute_error(test, pred_auto_sarima)
rmse_auto_sarima = np.sqrt(mean_squared_error(test, pred_auto_sarima))

print(f"MAE - Auto-SARIMA: {mae_auto_sarima:.2f}")
print(f"RMSE - Auto-SARIMA: {rmse_auto_sarima:.2f}")

#### Seasonal Period: 12 Months

In [ ]:
from pmdarima import auto_arima

modelo_arima = auto_arima(
    train,
    seasonal=True,
    m=12,
    stepwise=False,
    trace=True,
    suppress_warnings=True,
    error_action="ignore"
)

print(modelo_arima.summary())

In [ ]:
pred_auto_sarima = pd.Series(
    modelo_arima.predict(n_periods=len(test)),
    index=test.index
)

mae_auto_sarima = mean_absolute_error(test, pred_auto_sarima)
rmse_auto_sarima = np.sqrt(mean_squared_error(test, pred_auto_sarima))

print(f"MAE - Auto-SARIMA: {mae_auto_sarima:.2f}")
print(f"RMSE - Auto-SARIMA: {rmse_auto_sarima:.2f}")

**Conclusion:** Auto-SARIMA selected a model without a meaningful seasonal component. Although the decomposition showed visible seasonality, adding a seasonal structure did not improve the information criterion enough to justify the extra complexity.

### Model Diagnostics

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import jarque_bera

def diagnostico_modelo(nome, resid, aic):
    resid = pd.Series(resid).dropna()

    lb = acorr_ljungbox(resid, lags=[12], return_df=True)
    jb_stat, jb_pvalue, skew, kurtosis = jarque_bera(resid)

    print(f"Modelo: {nome}")
    print(f"AIC: {aic:.3f}")
    print(f"Ljung-Box p-value (lag 12): {lb['lb_pvalue'].iloc[0]:.4f}")
    print(f"Jarque-Bera p-value: {jb_pvalue:.4f}")
    print(f"Mean residual: {resid.mean():.4f}")
    print(f"Residual standard deviation: {resid.std():.4f}")
    print()

# ARIMA(2,1,1)
resid_arima_211 = ajuste_arima_211.resid
diagnostico_modelo(
    "ARIMA(2,1,1)",
    resid_arima_211,
    ajuste_arima_211.aic
)

# Auto-SARIMA
resid_auto_sarima = modelo_arima.resid()
diagnostico_modelo(
    "Auto-SARIMA",
    resid_auto_sarima,
    modelo_arima.aic()
)

**Conclusion:** The residual diagnostics compare autocorrelation and normality using Ljung-Box and Jarque-Bera tests. Final model selection should prioritize out-of-sample MAE and RMSE, while using AIC and residual diagnostics as supporting evidence.